# Support ticket preprocessing - Python and R
Solutions to all nine assignment tasks in one notebook. The Python kernel runs actual R code through **rpy2** (`%%R`). Each section includes the procedure, tables, and interpretation.

Data: `data/tickets_train.csv` (240 rows) and `data/tickets_validation.csv` (80 rows). Identifiers and original values are preserved. Validation data never fit preprocessing parameters; validation labels are never used. No classifier is trained because the assignment does not require one.

**Running the notebook:** Python 3.12+, an R installation, and `pandas numpy scipy scikit-learn rpy2 ipython ipykernel nbformat`. If needed, install packages in the kernel with `%pip install pandas numpy scipy scikit-learn rpy2 ipykernel`, restart the kernel, and select Run All. On Windows, R is located automatically under Program Files; elsewhere, set `R_HOME` if necessary. R uses only base packages and `Matrix`, which is usually included with R.

Both branches use numerical imputers from scikit-learn: R data are transferred to Python through rpy2, models are fitted independently, and results are returned to R. This uses the exact requested `KNNImputer` and `IterativeImputer`. Other R steps are implemented directly in R.

In [1]:
from pathlib import Path
import os, sys, importlib.util
# Use project-local dependencies when available; otherwise use the kernel environment.
for root in (Path.cwd(), *Path.cwd().parents):
    deps = root / '.notebook_deps'
    if deps.is_dir():
        sys.path.insert(0, str(deps))
        break
if not os.environ.get('R_HOME') and os.name == 'nt':
    candidates = sorted(Path('C:/Program Files/R').glob('R-*'))
    if candidates:
        os.environ['R_HOME'] = str(candidates[-1])
os.environ.setdefault('RPY2_CFFI_MODE', 'ABI')
_r_dll_handles = []
if os.name == 'nt' and os.environ.get('R_HOME'):
    for sub in ('bin/x64', 'bin'):
        location = Path(os.environ['R_HOME']) / sub
        if location.is_dir():
            _r_dll_handles.append(os.add_dll_directory(str(location)))
            os.environ['PATH'] = str(location) + os.pathsep + os.environ['PATH']
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import sparse
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter
%load_ext rpy2.ipython
pd.set_option('display.max_colwidth', 90)
pd.set_option('display.max_rows', 300)
BASE = Path.cwd()
if not (BASE / 'data/tickets_train.csv').exists():
    BASE = BASE / '22_10_2026'
assert (BASE / 'data/tickets_train.csv').is_file(), 'Run the notebook from 22_10_2026 or the project root.'
OUT = BASE / 'vystup'
OUT.mkdir(exist_ok=True)
ro.globalenv['data_dir'] = str(BASE / 'data').replace('\\', '/')
print('Python:', sys.version.split()[0], '| R:', ro.r('R.version.string')[0])
print('Data:', BASE)

Python: 3.12.14 | R: R version 4.6.1 (2026-06-24 ucrt)
Data: D:\Programko\UJEP\PYR\22_10_2026


## 1. Loading, profiling, and auditing - Python
All columns are loaded as strings. Empty CSV fields are missing values; whitespace-only strings are counted separately. The audit is extended throughout the notebook and includes the split. Raw tables are preserved.

In [2]:
raw = {s: pd.read_csv(BASE / f'data/tickets_{s}.csv', dtype='string', keep_default_na=False,
                       na_values=['']) for s in ('train', 'validation')}
assert [len(raw[s]) for s in raw] == [240, 80]
audit_rows = []
def audit(split, frame, col, mask, issue):
    for idx in frame.index[pd.Series(mask, index=frame.index).fillna(False)]:
        audit_rows.append(dict(split=split, ticket_id=frame.at[idx, 'ticket_id'], column=col,
                               original_value=frame.at[idx, col], issue_type=issue))
def profile(frame):
    return pd.DataFrame([dict(column=c, dtype=str(frame[c].dtype), missing=frame[c].isna().sum(),
        whitespace_only=frame[c].str.fullmatch(r'\s+').fillna(False).sum(),
        unique_nonmissing=frame[c].nunique(dropna=True)) for c in frame])
for s, df in raw.items():
    print(s); display(profile(df))
    for c in df:
        audit(s, df, c, df[c].isna(), 'missing')
        audit(s, df, c, df[c].str.fullmatch(r'\s+'), 'whitespace_only')

train


,column,dtype,missing,whitespace_only,unique_nonmissing
0,ticket_id,string,0,0,240
1,customer_id,string,0,0,60
2,created_at,string,0,0,234
3,first_reply_at,string,0,0,240
4,channel,string,9,0,4
5,priority,string,7,0,4
6,city,string,0,0,5
7,message,string,21,10,17
8,customer_age,string,26,0,58
9,order_value_eur,string,17,0,210


validation


,column,dtype,missing,whitespace_only,unique_nonmissing
0,ticket_id,string,0,0,80
1,customer_id,string,0,0,20
2,created_at,string,0,0,79
3,first_reply_at,string,0,0,80
4,channel,string,3,0,5
5,priority,string,3,0,4
6,city,string,0,0,6
7,message,string,7,3,14
8,customer_age,string,9,0,42
9,order_value_eur,string,5,0,73


## 2. Normalizing categories - Python
Trim and normalize whitespace and case. Both `Ústí nad Labem` and `Usti nad Labem` map to `usti nad labem`. The file also contains a corrupted variant with U+FFFD (`�st� nad Labem`); only this specific observed alias is repaired and audited. No general spelling guesses are made. Invalid priority `urgent` becomes missing; the declared order is low < medium < high. New valid nominal categories are retained.

In [3]:
def norm(series):
    return series.str.strip().str.replace(r'\s+', ' ', regex=True).str.lower().replace('', pd.NA)
clean = {}
frequencies = []
unseen = {}
for s, df in raw.items():
    d = df.copy()
    for c in ['channel', 'city', 'priority']:
        d[c + '_clean'] = norm(df[c])
    city_aliases = {'ústí nad labem': 'usti nad labem', '�st� nad labem': 'usti nad labem'}
    audit(s, df, 'city', d.city_clean.eq('�st� nad labem'), 'known_corrupted_city_alias')
    d['city_clean'] = d.city_clean.replace(city_aliases)
    invalid = d.priority_clean.notna() & ~d.priority_clean.isin(['low','medium','high'])
    audit(s, df, 'priority', invalid, 'invalid_priority_to_missing')
    d.loc[invalid, 'priority_clean'] = pd.NA
    for c in ['channel', 'city', 'priority']:
        for stage, x in [('raw', df[c]), ('clean', d[c+'_clean'])]:
            freq = x.fillna('<MISSING>').value_counts().rename_axis('category').reset_index(name='count')
            freq = freq.assign(split=s, column=c, stage=stage)
            frequencies.append(freq)
    clean[s] = d
freq_table = pd.concat(frequencies, ignore_index=True)
display(freq_table)
for c in ['channel', 'city', 'priority']:
    unseen[c] = sorted(set(clean['validation'][c+'_clean'].dropna()) - set(clean['train'][c+'_clean'].dropna()))
display(pd.DataFrame({'column': unseen.keys(), 'unseen_validation': unseen.values()}))

,category,count,split,column,stage
0,web,117,train,channel,raw
1,email,68,train,channel,raw
2,phone,28,train,channel,raw
3,EMAIL,18,train,channel,raw
4,<MISSING>,9,train,channel,raw
5,web,117,train,channel,clean
6,email,86,train,channel,clean
7,phone,28,train,channel,clean
8,<MISSING>,9,train,channel,clean
9,Usti nad Labem,67,train,city,raw


,column,unseen_validation
0,channel,[chat]
1,city,[liberec]
2,priority,[]


## 3. Timestamps and leakage prevention - Python
Offset-aware timestamps are converted to UTC. Local creation hour and weekday use Europe/Prague (Monday = 0). Invalid-time indicators are retained; no original timestamp or missing hour is invented. Later imputation of time features serves numerical algorithms only. Response duration is an audit-only field.

In [4]:
for s, d in clean.items():
    for c in ['created_at', 'first_reply_at']:
        d[c+'_utc'] = pd.to_datetime(d[c], utc=True, errors='coerce', format='ISO8601')
        bad = d[c].notna() & d[c+'_utc'].isna()
        audit(s, raw[s], c, bad, 'timestamp_parse_failed')
        print(s, c, 'failed conversions:', int(bad.sum()))
    local = d.created_at_utc.dt.tz_convert('Europe/Prague')
    d['created_invalid'] = (d.created_at.notna() & d.created_at_utc.isna()).astype(int)
    d['created_missing'] = d.created_at.isna().astype(int)
    d['creation_hour'] = local.dt.hour.astype(float)
    d['creation_weekday'] = local.dt.dayofweek.astype(float)
    d['hour_sin'] = np.sin(2*np.pi*d.creation_hour/24)
    d['hour_cos'] = np.cos(2*np.pi*d.creation_hour/24)
    d['response_minutes'] = (d.first_reply_at_utc-d.created_at_utc).dt.total_seconds()/60
    d['negative_response'] = (d.response_minutes < 0).astype(int)
    audit(s, raw[s], 'first_reply_at', d.response_minutes.lt(0), 'negative_response_duration')
    display(d[['ticket_id','created_at','created_at_utc','creation_hour','creation_weekday',
               'created_invalid','response_minutes','negative_response']].head(8))

train created_at failed conversions: 7
train first_reply_at failed conversions: 0


,ticket_id,created_at,created_at_utc,creation_hour,creation_weekday,created_invalid,response_minutes,negative_response
0,T0000,2026-02-30T10:00:00+01:00,NaT,NaN,NaN,1,NaN,0
1,T0001,2026-05-14T12:30:00+02:00,2026-05-14 10:30:00+00:00,12.0,3.0,0,143.0,0
2,T0002,2026-04-07T12:27:00+02:00,2026-04-07 10:27:00+00:00,12.0,1.0,0,59.0,0
3,T0003,2026-05-01T06:02:00+02:00,2026-05-01 04:02:00+00:00,6.0,4.0,0,232.0,0
4,T0004,2026-04-12T12:15:00+02:00,2026-04-12 10:15:00+00:00,12.0,6.0,0,157.0,0
5,T0005,2026-03-05T16:17:00+01:00,2026-03-05 15:17:00+00:00,16.0,3.0,0,198.0,0
6,T0006,2026-05-05T05:59:00+02:00,2026-05-05 03:59:00+00:00,5.0,1.0,0,176.0,0
7,T0007,2026-01-30T18:22:00+01:00,2026-01-30 17:22:00+00:00,18.0,4.0,0,189.0,0


validation created_at failed conversions: 2
validation first_reply_at failed conversions: 0


,ticket_id,created_at,created_at_utc,creation_hour,creation_weekday,created_invalid,response_minutes,negative_response
0,T0240,2026-04-17T02:27:00+02:00,2026-04-17 00:27:00+00:00,2.0,4.0,0,141.0,0
1,T0241,2026-01-21T00:47:00+01:00,2026-01-20 23:47:00+00:00,0.0,2.0,0,264.0,0
2,T0242,2026-05-23T11:05:00+02:00,2026-05-23 09:05:00+00:00,11.0,5.0,0,131.0,0
3,T0243,2026-06-22T11:54:00+02:00,2026-06-22 09:54:00+00:00,11.0,0.0,0,240.0,0
4,T0244,2026-06-02T21:02:00+02:00,2026-06-02 19:02:00+00:00,21.0,1.0,0,31.0,0
5,T0245,2026-02-28T07:34:00+01:00,2026-02-28 06:34:00+00:00,7.0,5.0,0,52.0,0
6,T0246,2026-02-08T12:21:00+01:00,2026-02-08 11:21:00+00:00,12.0,6.0,0,-30.0,1
7,T0247,2026-06-12T05:15:00+02:00,2026-06-12 03:15:00+00:00,5.0,4.0,0,17.0,0


## 4. Missing text and categories - Python
Indicators are derived from cleaned fields. Unavailable text becomes `""`; negation and message content are preserved. The most frequent category is learned on training data only; the second strategy uses the reserved category `__MISSING__`. The explicit missing category is selected for subsequent steps so unavailable channels remain distinguishable from the most frequent channel.

In [5]:
MISSING = '__MISSING__'
cat_columns = ['channel_clean','city_clean','priority_clean']
modes = {c: clean['train'][c].mode().iloc[0] for c in cat_columns}
channel_proportions = []
for s, d in clean.items():
    d['message_clean'] = d.message.fillna('').str.strip()
    d['message_missing'] = d.message_clean.eq('').astype(int)
    for c in ['channel','priority']:
        d[c+'_missing'] = d[c+'_clean'].isna().astype(int)
    for c in cat_columns:
        assert not d[c].eq(MISSING).fillna(False).any()
        d[c+'_mode'] = d[c].fillna(modes[c])
        d[c+'_explicit'] = d[c].fillna(MISSING)
    for strategy in ['mode','explicit']:
        prop = d['channel_clean_'+strategy].value_counts(normalize=True).rename('proportion').reset_index()
        prop.columns = ['channel','proportion']
        channel_proportions.append(prop.assign(split=s, strategy=strategy))
display(pd.concat(channel_proportions, ignore_index=True))

,channel,proportion,split,strategy
0,web,0.525,train,mode
1,email,0.358333,train,mode
2,phone,0.116667,train,mode
3,web,0.4875,train,explicit
4,email,0.358333,train,explicit
5,phone,0.116667,train,explicit
6,__MISSING__,0.0375,train,explicit
7,email,0.425,validation,mode
8,web,0.3375,validation,mode
9,phone,0.1375,validation,mode


## 5. Numerical domains and imputation comparison - Python
Ages outside [18, 100] and negative amounts violate the domain. A large positive amount is not automatically an error. Separate indicators retain original missingness, conversion failures, and domain violations.

With seed 42, hide 20% of originally observed, domain-valid values in each of the two validation columns. Every method receives the same mask. KNN follows standardization fitted on incomplete training data. IterativeImputer uses its default BayesianRidge estimator, 20 iterations, and a fixed seed. Models fit incomplete training data only; no input contains an unmasked copy of the hidden values. Score artificial gaps only, excluding naturally missing entries. Median imputation is selected in advance for final preprocessing as a simple, stable baseline.

In [6]:
num_cols = ['customer_age_num','order_value_eur_num']
time_cols = ['creation_hour','creation_weekday','hour_sin','hour_cos']
impute_cols = num_cols + time_cols
for s, d in clean.items():
    for c in ['customer_age','order_value_eur']:
        values = pd.to_numeric(d[c], errors='coerce').astype(float)
        missing = d[c].isna() | d[c].str.strip().eq('').fillna(False)
        failure = ~missing & values.isna()
        invalid = (values.lt(18) | values.gt(100)) if c == 'customer_age' else values.lt(0)
        invalid |= values.notna() & ~np.isfinite(values)
        d[c+'_missing'] = missing.astype(int)
        d[c+'_parse_failed'] = failure.astype(int)
        d[c+'_domain_invalid'] = invalid.astype(int)
        d[c+'_num'] = values.mask(invalid)
        audit(s, raw[s], c, failure, 'numeric_parse_failed')
        audit(s, raw[s], c, invalid, 'domain_violation')
def compare_imputers(train, validation, seed=42):
    tr = train[impute_cols].astype(float).to_numpy(copy=True)
    truth = validation[impute_cols].astype(float).to_numpy(copy=True)
    masked = truth.copy()
    mask = np.zeros_like(truth, dtype=bool)
    rng = np.random.default_rng(seed)
    for j in range(2):
        observed = np.flatnonzero(np.isfinite(truth[:,j]))
        chosen = rng.choice(observed, size=max(1, int(len(observed)*.2)), replace=False)
        mask[chosen,j] = True
    masked[mask] = np.nan
    models = {'median': SimpleImputer(strategy='median'),
              'scaled_KNN': make_pipeline(StandardScaler(), KNNImputer(n_neighbors=5)),
              'IterativeImputer': IterativeImputer(random_state=42, max_iter=20)}
    scores, reconstructed = [], {}
    for name, model in models.items():
        model.fit(tr)
        pred = model.transform(masked)
        if name == 'scaled_KNN':
            pred = model.named_steps['standardscaler'].inverse_transform(pred)
        reconstructed[name] = pred
        for j, unit in enumerate(['years','EUR']):
            m = mask[:,j]
            scores.append(dict(method=name, feature=num_cols[j], unit=unit, n_hidden=int(m.sum()),
                               MAE=float(np.abs(pred[m,j]-truth[m,j]).mean())))
    return pd.DataFrame(scores), mask, reconstructed
imputer_scores, artificial_mask, reconstruction = compare_imputers(clean['train'], clean['validation'])
display(imputer_scores)
chosen_imputer = SimpleImputer(strategy='median').fit(clean['train'][impute_cols])
imputed = {s: pd.DataFrame(chosen_imputer.transform(d[impute_cols]), columns=impute_cols, index=d.index)
           for s,d in clean.items()}
display(pd.concat([d[[c for c in d if c.endswith(('_parse_failed','_domain_invalid','_missing'))]].sum().rename(s)
                   for s,d in clean.items()], axis=1))

,method,feature,unit,n_hidden,MAE
0,median,customer_age_num,years,13,16.307692
1,median,order_value_eur_num,EUR,14,90.075714
2,scaled_KNN,customer_age_num,years,13,14.107692
3,scaled_KNN,order_value_eur_num,EUR,14,520.482286
4,IterativeImputer,customer_age_num,years,13,16.426010
5,IterativeImputer,order_value_eur_num,EUR,14,276.163665


,train,validation
created_missing,0,0
message_missing,31,10
channel_missing,9,3
priority_missing,11,4
customer_age_missing,26,9
customer_age_parse_failed,0,0
customer_age_domain_invalid,6,2
order_value_eur_missing,17,5
order_value_eur_parse_failed,4,2
order_value_eur_domain_invalid,5,1


## 6. Unusual observations - Python
Tukey thresholds Q1 - 1.5 IQR and Q3 + 1.5 IQR are fitted on training data only. Amounts are reviewed after domain validation. Text length counts characters in trimmed messages; unavailable messages are excluded from the length distribution. Review does not imply exclusion: large orders, long reports, and rare categories are retained.

In [7]:
thresholds, review_rows = {}, []
for d in clean.values():
    d['message_length'] = d.message_clean.str.len().astype(float).mask(d.message_missing.eq(1))
for c in ['order_value_eur_num','message_length']:
    q1,q3 = clean['train'][c].quantile([.25,.75])
    thresholds[c] = (q1-1.5*(q3-q1), q3+1.5*(q3-q1))
    lo, hi = thresholds[c]
    for s,d in clean.items():
        unusual = d[c].lt(lo) | d[c].gt(hi)
        d[c+'_unusual'] = unusual.astype(int)
        for i in d.index[unusual]:
            review_rows.append(dict(split=s, ticket_id=d.at[i,'ticket_id'], feature=c,
                value=d.at[i,c], lower=lo, upper=hi, reason='IQR_review_potentially_valid', decision='keep'))
for s,d in clean.items():
    for c,lo,hi in [('customer_age',18,100),('order_value_eur',0,np.inf)]:
        for i in d.index[d[c+'_domain_invalid'].eq(1)]:
            review_rows.append(dict(split=s,ticket_id=d.at[i,'ticket_id'],feature=c,value=d.at[i,c],
                lower=lo,upper=hi,reason='known_domain_violation',decision='working_value_to_missing'))
review = pd.DataFrame(review_rows)
display(pd.DataFrame(thresholds,index=['lower','upper']).T)
display(review)

,lower,upper
order_value_eur_num,-78.4475,235.4525
message_length,14.0000,62.0000


,split,ticket_id,feature,value,lower,upper,reason,decision
0,train,T0008,order_value_eur_num,610.85,-78.4475,235.4525,IQR_review_potentially_valid,keep
1,train,T0012,order_value_eur_num,300.67,-78.4475,235.4525,IQR_review_potentially_valid,keep
2,train,T0029,order_value_eur_num,267.13,-78.4475,235.4525,IQR_review_potentially_valid,keep
3,train,T0053,order_value_eur_num,15000.0,-78.4475,235.4525,IQR_review_potentially_valid,keep
4,train,T0057,order_value_eur_num,791.42,-78.4475,235.4525,IQR_review_potentially_valid,keep
5,train,T0076,order_value_eur_num,291.49,-78.4475,235.4525,IQR_review_potentially_valid,keep
6,train,T0106,order_value_eur_num,15000.0,-78.4475,235.4525,IQR_review_potentially_valid,keep
7,train,T0155,order_value_eur_num,319.18,-78.4475,235.4525,IQR_review_potentially_valid,keep
8,train,T0157,order_value_eur_num,267.64,-78.4475,235.4525,IQR_review_potentially_valid,keep
9,train,T0159,order_value_eur_num,15000.0,-78.4475,235.4525,IQR_review_potentially_valid,keep


## 7. Numerical scaling - Python
Compare only the two imputed numerical features. StandardScaler uses the mean and population standard deviation, MinMaxScaler uses the training range, and RobustScaler uses the median and IQR. To measure the largest order's influence, refit each scaler without that single training row and compare transformations of the remaining training rows. This is a sensitivity diagnostic; the main scalers and data retain the largest order.

In [8]:
def scale_comparison(train, validation):
    scalers = {'standard':StandardScaler(), 'minmax':MinMaxScaler(), 'robust':RobustScaler()}
    tables, transformed, influence = [], {}, []
    ix = int(np.argmax(train.order_value_eur_num.to_numpy()))
    keep = np.arange(len(train)) != ix
    for name, scaler in scalers.items():
        scaler.fit(train)
        transformed[name] = scaler.transform(validation)
        for split, a in [('train',scaler.transform(train)),('validation',transformed[name])]:
            tables.append(pd.DataFrame(a,columns=num_cols).describe().T.assign(method=name,split=split))
        alternative = type(scaler)().fit(train.iloc[keep])
        shift = np.abs(scaler.transform(train.iloc[keep])-alternative.transform(train.iloc[keep])).mean(axis=0)
        for j,c in enumerate(num_cols):
            influence.append(dict(method=name,feature=c,mean_abs_shift_without_largest_order=shift[j]))
    return pd.concat(tables), transformed, pd.DataFrame(influence)
display(pd.concat([d[num_cols].describe().T.assign(split=s) for s,d in imputed.items()]))
scale_summary, scaled_validation, scale_influence = scale_comparison(imputed['train'][num_cols], imputed['validation'][num_cols])
display(scale_summary)
display(pd.concat({k:pd.DataFrame(v,columns=num_cols).assign(ticket_id=clean['validation'].ticket_id)
                   for k,v in scaled_validation.items()}, names=['method','row']))
outside = (imputed['validation'][num_cols].lt(imputed['train'][num_cols].min()) |
           imputed['validation'][num_cols].gt(imputed['train'][num_cols].max()))
display(pd.concat([clean['validation'][['ticket_id']], imputed['validation'][num_cols], outside.add_suffix('_outside_train')],axis=1).loc[outside.any(axis=1)])
display(scale_influence)

,count,mean,std,min,25%,50%,75%,max,split
customer_age_num,240.0,49.754167,16.773222,18.00,36.000,51.00,65.0000,77.0,train
order_value_eur_num,240.0,337.535750,1914.891368,9.82,43.080,62.11,106.5375,15000.0,train
customer_age_num,80.0,48.375000,15.185249,18.00,38.750,51.00,60.0000,77.0,validation
order_value_eur_num,80.0,472.665875,2342.966384,7.49,44.585,65.50,125.1575,15000.0,validation


,count,mean,std,min,25%,50%,75%,max,method,split
customer_age_num,240.0,-1.933638e-16,1.002090,-1.897103,-0.821721,0.074430,0.910838,1.627760,standard,train
order_value_eur_num,240.0,3.700743e-18,1.002090,-0.171498,-0.154093,-0.144134,-0.120885,7.673076,standard,train
customer_age_num,80.0,-8.239615e-02,0.907219,-1.897103,-0.657427,0.074430,0.612121,1.627760,standard,validation
order_value_eur_num,80.0,7.071551e-02,1.226108,-0.172718,-0.153305,-0.142360,-0.111141,7.673076,standard,validation
customer_age_num,240.0,5.382062e-01,0.284292,0.000000,0.305085,0.559322,0.796610,1.000000,minmax,train
order_value_eur_num,240.0,2.186203e-02,0.127743,0.000000,0.002219,0.003488,0.006452,1.000000,minmax,train
customer_age_num,80.0,5.148305e-01,0.257377,0.000000,0.351695,0.559322,0.711864,1.000000,minmax,validation
order_value_eur_num,80.0,3.087661e-02,0.156300,-0.000155,0.002319,0.003714,0.007694,1.000000,minmax,validation
customer_age_num,240.0,-4.295977e-02,0.578387,-1.137931,-0.517241,0.000000,0.482759,0.896552,robust,train
order_value_eur_num,240.0,4.340318e+00,30.175966,-0.824016,-0.299886,0.000000,0.700114,235.399913,robust,train


customer_age_num  order_value_eur_num ticket_id
method   row                                                 
standard 0           -1.120438            -0.157275     T0240
         1            0.074430            -0.166155     T0241
         2            0.851095            -0.112280     T0242
         3            0.074430            -0.131245     T0243
         4            0.910838            -0.144134     T0244
         5           -0.224287            -0.163026     T0245
         6           -1.180182            -0.094712     T0246
         7            1.568016            -0.139000     T0247
         8           -0.702234            -0.142229     T0248
         9            0.970582            -0.122385     T0249
         10           0.492634            -0.154930     T0250
         11          -1.777616             0.051543     T0251
         12           0.074430            -0.144134     T0252
         13           0.791352            -0.132245     T0253
         14           0.373148            -0.154747     T0254
         15           0.851095            -0.141261     T0255
         16          -0.104800            -0.073304     T0256
         17          -0.343774            -0.071247     T0257
         18           0.074430            -0.128717     T0258
         19           0.193917            -0.130319     T0259
         20          -0.224287            -0.163769     T0260
         21           0.074430            -0.141952     T0261
         22          -1.120438            -0.127472     T0262
         23          -1.658129            -0.142653     T0263
         24          -1.359412            -0.104174     T0264
         25           0.134174             7.673076     T0265
         26           0.492634            -0.144134     T0266
         27           1.030325             0.132307     T0267
         28          -0.642491            -0.109166     T0268
         29          -1.837359            -0.168165     T0269
         30           0.074430            -0.163246     T0270
         31           1.030325            -0.132339     T0271
         32          -0.463261            -0.143930     T0272
         33          -0.523004            -0.080902     T0273
         34          -1.777616            -0.143590     T0274
         35           0.253661            -0.113321     T0275
         36           0.612121            -0.074005     T0276
         37          -0.343774            -0.166386     T0277
         38          -0.463261            -0.002813     T0278
         39           0.074430            -0.100767     T0279
         40          -1.897103            -0.144134     T0280
         41           1.329042            -0.133720     T0281
         42          -1.538642            -0.144134     T0282
         43           0.134174            -0.164355     T0283
         44           0.612121            -0.155281     T0284
         45           0.910838            -0.135280     T0285
         46           1.090069            -0.146971     T0286
         47          -1.180182            -0.067547     T0287
         48           0.074430            -0.142491     T0288
         49           1.388786            -0.134500     T0289
         50           0.134174            -0.150158     T0290
         51          -0.104800            -0.145066     T0291
         52          -0.284030            -0.091300     T0292
         53          -0.224287            -0.146081     T0293
         54           1.627760            -0.144134     T0294
         55          -1.000951            -0.066532     T0295
         56           0.552378            -0.158206     T0296
         57           0.074430            -0.172718     T0297
         58          -1.598386            -0.161226     T0298
         59          -0.523004            -0.170514     T0299
         60           0.791352            -0.152915     T0300
         61           0.074430            -0.138315     T0301
         62          -0.403517            -0.1

,ticket_id,customer_age_num,order_value_eur_num,customer_age_num_outside_train,order_value_eur_num_outside_train
57,T0297,51.0,7.49,False,True


,method,feature,mean_abs_shift_without_largest_order
0,standard,customer_age_num,0.006188
1,standard,order_value_eur_num,0.032105
2,minmax,customer_age_num,0.000000
3,minmax,order_value_eur_num,0.000000
4,robust,customer_age_num,0.000000
5,robust,order_value_eur_num,0.034370


## 8. Consistent categorical encoding - Python
Nominal fields use one-hot encoding with explicit missing values. The chosen `min_frequency=30` threshold groups categories with fewer than 30 training observations. `ignore` produces a zero block for an unknown category; `infrequent_if_exist` maps it to the infrequent bucket only if that field has one. Priority uses low=0, medium=1, high=2, and missing/invalid=-1 with a separate indicator.

In [9]:
nominal_cols = ['channel_clean_explicit','city_clean_explicit']
encoders, encoded = {}, {}
for policy in ['ignore','infrequent_if_exist']:
    enc = OneHotEncoder(handle_unknown=policy, min_frequency=30 if policy=='infrequent_if_exist' else None,
                        sparse_output=True)
    enc.fit(clean['train'][nominal_cols])
    matrices = {s:enc.transform(d[nominal_cols]) for s,d in clean.items()}
    assert matrices['train'].shape[1] == matrices['validation'].shape[1]
    encoders[policy], encoded[policy] = enc, matrices
    print(policy, {s:m.shape for s,m in matrices.items()})
    print('Features:', enc.get_feature_names_out().tolist())
    if policy == 'infrequent_if_exist':
        display(pd.DataFrame({'field':nominal_cols,
            'infrequent_bucket_exists':[v is not None and len(v)>0 for v in enc.infrequent_categories_],
            'infrequent_training_categories':[list(v) if v is not None else [] for v in enc.infrequent_categories_]}))
    v = clean['validation']
    unknown = np.zeros(len(v), dtype=bool)
    for c in nominal_cols:
        unknown |= ~v[c].isin(clean['train'][c].unique()).to_numpy()
    ids = np.flatnonzero(unknown)[:8]
    display(pd.concat([v.iloc[ids][['ticket_id']+nominal_cols].reset_index(drop=True),
                       pd.DataFrame(matrices['validation'][ids].toarray(), columns=enc.get_feature_names_out())],axis=1))
for d in clean.values():
    d['priority_code'] = d.priority_clean.map({'low':0,'medium':1,'high':2}).fillna(-1).astype(int)
display(clean['validation'][['ticket_id','priority','priority_clean','priority_code','priority_missing']].head(10))

ignore {'train': (240, 8), 'validation': (80, 8)}
Features: ['channel_clean_explicit___MISSING__', 'channel_clean_explicit_email', 'channel_clean_explicit_phone', 'channel_clean_explicit_web', 'city_clean_explicit_brno', 'city_clean_explicit_decin', 'city_clean_explicit_prague', 'city_clean_explicit_usti nad labem']


,ticket_id,channel_clean_explicit,city_clean_explicit,channel_clean_explicit___MISSING__,channel_clean_explicit_email,channel_clean_explicit_phone,channel_clean_explicit_web,city_clean_explicit_brno,city_clean_explicit_decin,city_clean_explicit_prague,city_clean_explicit_usti nad labem
0,T0241,phone,liberec,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,T0244,phone,liberec,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,T0249,web,liberec,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,T0250,chat,prague,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,T0258,web,liberec,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,T0259,email,liberec,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,T0260,email,liberec,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7,T0263,web,liberec,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


infrequent_if_exist {'train': (240, 7), 'validation': (80, 7)}
Features: ['channel_clean_explicit_email', 'channel_clean_explicit_web', 'channel_clean_explicit_infrequent_sklearn', 'city_clean_explicit_brno', 'city_clean_explicit_decin', 'city_clean_explicit_prague', 'city_clean_explicit_usti nad labem']


,field,infrequent_bucket_exists,infrequent_training_categories
0,channel_clean_explicit,True,"[__MISSING__, phone]"
1,city_clean_explicit,False,[]


,ticket_id,channel_clean_explicit,city_clean_explicit,channel_clean_explicit_email,channel_clean_explicit_web,channel_clean_explicit_infrequent_sklearn,city_clean_explicit_brno,city_clean_explicit_decin,city_clean_explicit_prague,city_clean_explicit_usti nad labem
0,T0241,phone,liberec,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,T0244,phone,liberec,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,T0249,web,liberec,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,T0250,chat,prague,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,T0258,web,liberec,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5,T0259,email,liberec,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,T0260,email,liberec,1.0,0.0,0.0,0.0,0.0,0.0,0.0
7,T0263,web,liberec,0.0,1.0,0.0,0.0,0.0,0.0,0.0


,ticket_id,priority,priority_clean,priority_code,priority_missing
0,T0240,low,low,0,0
1,T0241,low,low,0,0
2,T0242,medium,medium,1,0
3,T0243,medium,medium,1,0
4,T0244,medium,medium,1,0
5,T0245,low,low,0,0
6,T0246,medium,medium,1,0
7,T0247,low,low,0,0
8,T0248,<NA>,<NA>,-1,1
9,T0249,medium,medium,1,0


## 9. TF-IDF - Python
No stop-word filtering, stemming, or negation removal is applied. Tokens contain at least two word characters, matching scikit-learn's default word tokenization. Both representations fit training text only. The selected unigram/bigram representation stays sparse; illustrative phrases are transformed only.

In [10]:
phrases = ['payment problem', 'no payment problem', 'order has arrived', 'order has not arrived']
text_models, text_matrices, text_stats = {}, {}, []
for name, ngram in [('unigram',(1,1)), ('unigram_bigram',(1,2))]:
    vec = TfidfVectorizer(ngram_range=ngram, lowercase=True, stop_words=None)
    vec.fit(clean['train'].message_clean)
    matrices = {s:vec.transform(d.message_clean).tocsr() for s,d in clean.items()}
    text_models[name], text_matrices[name] = vec, matrices
    for s,m in matrices.items():
        text_stats.append(dict(representation=name, split=s, rows=m.shape[0], columns=m.shape[1],
                               nnz=m.nnz, zero_vectors=int((m.getnnz(axis=1)==0).sum())))
    feature_names = vec.get_feature_names_out()
    probe = vec.transform(phrases)
    selected = np.flatnonzero(np.asarray(probe.getnnz(axis=0)).ravel()>0)
    print(name, '- nonzero weights for illustrative phrases:')
    display(pd.DataFrame(probe[:,selected].toarray(),index=phrases,columns=feature_names[selected]).round(4))
    print('Features containing negation:', [x for x in feature_names if 'not' in x.split() or 'no' in x.split()])
display(pd.DataFrame(text_stats))
# Final inputs exclude identifiers, targets, and reply information.
flag_cols = ['message_missing','channel_missing','priority_missing','created_invalid','created_missing'] + [
    c+s for c in ['customer_age','order_value_eur'] for s in ['_missing','_parse_failed','_domain_invalid']]
final_scaler = RobustScaler().fit(imputed['train'][num_cols])
X = {}
feature_names_final = (['age_robust','order_robust'] + time_cols + flag_cols + ['priority_code'] +
                      encoders['ignore'].get_feature_names_out().tolist() +
                      ['tfidf__'+x for x in text_models['unigram_bigram'].get_feature_names_out()])
for s,d in clean.items():
    dense = np.column_stack([final_scaler.transform(imputed[s][num_cols]), imputed[s][time_cols],
                             d[flag_cols].astype(float), d[['priority_code']]])
    X[s] = sparse.hstack([sparse.csr_matrix(dense),encoded['ignore'][s],
                          text_matrices['unigram_bigram'][s]], format='csr')
    assert sparse.issparse(X[s]) and np.isfinite(X[s].data).all()
assert X['train'].shape[1] == X['validation'].shape[1] == len(feature_names_final)
assert not any(c in feature_names_final for c in ['ticket_id','customer_id','needs_escalation','response_minutes','negative_response'])
print('Final sparse inputs:', {s:m.shape for s,m in X.items()})
audit_table = pd.DataFrame(audit_rows).drop_duplicates().reset_index(drop=True)
display(audit_table.groupby(['split','column','issue_type'],dropna=False).size().rename('count').reset_index())

unigram - nonzero weights for illustrative phrases:


,arrived,has,no,not,order,payment,problem
payment problem,0.0000,0.0000,0.0000,0.0000,0.0000,0.5477,0.8367
no payment problem,0.0000,0.0000,0.5573,0.0000,0.0000,0.4548,0.6947
order has arrived,0.5598,0.6664,0.0000,0.0000,0.4925,0.0000,0.0000
order has not arrived,0.5052,0.6015,0.0000,0.4305,0.4445,0.0000,0.0000


Features containing negation: ['no', 'not']
unigram_bigram - nonzero weights for illustrative phrases:


,arrived,has,has not,no,no payment,not,not arrived,order,order has,payment,payment problem,problem
payment problem,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.4201,0.6417,0.6417
no payment problem,0.0000,0.0000,0.0000,0.3976,0.4956,0.0000,0.0000,0.0000,0.0000,0.3244,0.4956,0.4956
order has arrived,0.4658,0.5546,0.0000,0.0000,0.0000,0.0000,0.0000,0.4098,0.5546,0.0000,0.0000,0.0000
order has not arrived,0.3499,0.4165,0.4165,0.0000,0.0000,0.2981,0.4165,0.3078,0.4165,0.0000,0.0000,0.0000


Features containing negation: ['did not', 'has not', 'have not', 'no', 'no further', 'no payment', 'not', 'not arrived', 'not help', 'not received', 'received no']


,representation,split,rows,columns,nnz,zero_vectors
0,unigram,train,240,50,1249,31
1,unigram,validation,80,50,423,10
2,unigram_bigram,train,240,113,2295,31
3,unigram_bigram,validation,80,113,777,10


Final sparse inputs: {'train': (240, 139), 'validation': (80, 139)}


,split,column,issue_type,count
0,train,channel,missing,9
1,train,created_at,timestamp_parse_failed,7
2,train,customer_age,domain_violation,6
3,train,customer_age,missing,26
4,train,first_reply_at,negative_response_duration,5
5,train,message,missing,21
6,train,message,whitespace_only,10
7,train,order_value_eur,domain_violation,5
8,train,order_value_eur,missing,17
9,train,order_value_eur,numeric_parse_failed,4


## R through rpy2: 1-4. Loading, auditing, categories, timestamps, and missingness
This branch reloads the original CSV files directly in R. `%%R` is an rpy2 magic command running in the Python kernel. Feature definitions match the Python branch. The strict parser also validates calendar dates by formatting them back, preventing an invalid date such as February 30 from silently rolling over.

In [11]:
%%R
options(width=130)
if (.Platform$OS.type == 'windows') invisible(Sys.setlocale('LC_CTYPE','English_United States.utf8'))
library(Matrix)
raw_r <- lapply(c(train='train',validation='validation'), function(s)
  read.csv(file.path(data_dir,paste0('tickets_',s,'.csv')), colClasses='character',
           na.strings='', fileEncoding='UTF-8', check.names=FALSE))
audit_r <- data.frame(split=character(),ticket_id=character(),column=character(),
                      original_value=character(),issue_type=character())
add_audit <- function(s,d,c,mask,issue) {
  ix <- which(!is.na(mask) & mask)
  if(length(ix)) audit_r <<- rbind(audit_r,data.frame(split=s,ticket_id=d$ticket_id[ix],
      column=c,original_value=d[[c]][ix],issue_type=issue))
}
profile_r <- function(d) do.call(rbind,lapply(names(d),function(c) data.frame(
  column=c,dtype=class(d[[c]])[1],missing=sum(is.na(d[[c]])),
  whitespace_only=sum(grepl('^[[:space:]]+$',d[[c]]),na.rm=TRUE),
  unique_nonmissing=length(unique(na.omit(d[[c]]))))))
normalize <- function(x) {
  x <- tolower(gsub('[[:space:]]+',' ',trimws(x)))
  x[!is.na(x) & x==''] <- NA_character_; x
}
parse_offset <- function(x) {
  result <- rep(as.POSIXct(NA,tz='UTC'),length(x))
  for(i in which(!is.na(x))) {
    if(!grepl('^[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}[+-][0-9]{2}:[0-9]{2}$',x[i])) next
    wall <- substr(x[i],1,19)
    z <- suppressWarnings(as.POSIXct(strptime(wall,'%Y-%m-%dT%H:%M:%S',tz='UTC')))
    if(is.na(z) || format(z,'%Y-%m-%dT%H:%M:%S',tz='UTC') != wall) next
    hh <- as.integer(substr(x[i],21,22)); mm <- as.integer(substr(x[i],24,25))
    if(hh>23 || mm>59) next
    sign <- if(substr(x[i],20,20)=='+') 1 else -1
    result[i] <- z-sign*(hh*3600+mm*60)
  }; result
}
clean_r <- list(); frequencies_r <- data.frame()
for(s in names(raw_r)) {
  d <- raw_r[[s]]
  cat('\nProfile:',s,'\n'); print(profile_r(d))
  for(c in names(d)) {
    add_audit(s,d,c,is.na(d[[c]]),'missing')
    add_audit(s,d,c,grepl('^[[:space:]]+$',d[[c]]),'whitespace_only')
  }
  for(c in c('channel','city','priority')) d[[paste0(c,'_clean')]] <- normalize(d[[c]])
  add_audit(s,d,'city',d$city_clean=='\ufffdst\ufffd nad labem','known_corrupted_city_alias')
  d$city_clean[d$city_clean %in% c('\u00fast\u00ed nad labem','\ufffdst\ufffd nad labem')] <- 'usti nad labem'
  bad <- !is.na(d$priority_clean) & !d$priority_clean %in% c('low','medium','high')
  add_audit(s,d,'priority',bad,'invalid_priority_to_missing'); d$priority_clean[bad] <- NA
  for(c in c('channel','city','priority')) for(stage in c('raw','clean')) {
    x <- d[[if(stage=='raw') c else paste0(c,'_clean')]]; x[is.na(x)] <- '<MISSING>'
    tab <- as.data.frame(table(x),stringsAsFactors=FALSE); names(tab) <- c('category','count')
    frequencies_r <- rbind(frequencies_r,cbind(split=s,column=c,stage=stage,tab))
  }
  for(c in c('created_at','first_reply_at')) {
    d[[paste0(c,'_utc')]] <- parse_offset(d[[c]])
    bad <- !is.na(d[[c]]) & is.na(d[[paste0(c,'_utc')]])
    add_audit(s,d,c,bad,'timestamp_parse_failed')
    cat(s,c,'failed:',sum(bad),'\n')
  }
  d$created_invalid <- as.integer(!is.na(d$created_at) & is.na(d$created_at_utc))
  d$created_missing <- as.integer(is.na(d$created_at))
  local <- as.POSIXlt(d$created_at_utc,tz='Europe/Prague')
  d$creation_hour <- as.numeric(local$hour)
  d$creation_weekday <- as.numeric((local$wday+6) %% 7)
  d$hour_sin <- sin(2*pi*d$creation_hour/24); d$hour_cos <- cos(2*pi*d$creation_hour/24)
  d$response_minutes <- as.numeric(difftime(d$first_reply_at_utc,d$created_at_utc,units='mins'))
  d$negative_response <- as.integer(!is.na(d$response_minutes) & d$response_minutes<0)
  add_audit(s,d,'first_reply_at',d$negative_response==1,'negative_response_duration')
  d$message_clean <- trimws(ifelse(is.na(d$message),'',d$message))
  d$message_missing <- as.integer(d$message_clean=='')
  for(c in c('channel','priority')) d[[paste0(c,'_missing')]] <- as.integer(is.na(d[[paste0(c,'_clean')]]))
  clean_r[[s]] <- d
}
print(frequencies_r,row.names=FALSE)
for(c in c('channel','city','priority')) {
  col <- paste0(c,'_clean')
  cat(c,'unseen:',setdiff(na.omit(clean_r$validation[[col]]),na.omit(clean_r$train[[col]])),'\n')
  counts <- table(clean_r$train[[col]]); mode <- names(counts)[which.max(counts)]
  for(s in names(clean_r)) {
    x <- clean_r[[s]][[col]]
    stopifnot(!any(x=='__MISSING__',na.rm=TRUE))
    clean_r[[s]][[paste0(col,'_mode')]] <- ifelse(is.na(x),mode,x)
    clean_r[[s]][[paste0(col,'_explicit')]] <- ifelse(is.na(x),'__MISSING__',x)
  }
}
for(s in names(clean_r)) for(strategy in c('mode','explicit')) {
  cat(s,strategy,'channel proportions\n')
  print(prop.table(table(clean_r[[s]][[paste0('channel_clean_',strategy)]])))
}
print(head(clean_r$validation[,c('ticket_id','creation_hour','creation_weekday','created_invalid','response_minutes')]))


Profile:

train

           column

     dtype

 missing

 whitespace_only

... (541 outputs left out)

                5

               0

               52

In addition: Warning message:
In Sys.setlocale("LC_CTYPE", "English_United States.utf8") :
  using locale code page other than 1250 may cause problems


## R: 5. Numerical domains and exact imputers through rpy2
R performs numerical validation and creates reason indicators. Only the six permitted numerical columns are transferred to the same scikit-learn function used above. The function fits fresh models on the R training table; results and the mask return to R. The imputation algorithms are deliberately shared rather than independently reimplemented in R.

In [12]:
%%R
num_cols_r <- c('customer_age_num','order_value_eur_num')
time_cols_r <- c('creation_hour','creation_weekday','hour_sin','hour_cos')
impute_cols_r <- c(num_cols_r,time_cols_r)
for(s in names(clean_r)) {
  d <- clean_r[[s]]
  for(c in c('customer_age','order_value_eur')) {
    x <- suppressWarnings(as.numeric(d[[c]]))
    missing <- is.na(d[[c]]) | trimws(d[[c]])==''; missing[is.na(missing)] <- TRUE
    failed <- !missing & is.na(x)
    invalid <- if(c=='customer_age') !is.na(x) & (x<18 | x>100) else !is.na(x) & x<0
    invalid <- invalid | (!is.na(x) & !is.finite(x))
    d[[paste0(c,'_missing')]] <- as.integer(missing)
    d[[paste0(c,'_parse_failed')]] <- as.integer(failed)
    d[[paste0(c,'_domain_invalid')]] <- as.integer(invalid)
    add_audit(s,d,c,failed,'numeric_parse_failed'); add_audit(s,d,c,invalid,'domain_violation')
    x[invalid] <- NA_real_; d[[paste0(c,'_num')]] <- x
  }
  clean_r[[s]] <- d
}
r_train_numeric <- clean_r$train[,impute_cols_r]
r_validation_numeric <- clean_r$validation[,impute_cols_r]

In [13]:
with localconverter(default_converter + pandas2ri.converter):
    r_tr = ro.conversion.rpy2py(ro.globalenv['r_train_numeric'])
    r_va = ro.conversion.rpy2py(ro.globalenv['r_validation_numeric'])
# R NA_real_ has a special NaN bit pattern; normalize it to NumPy NaN.
for frame in (r_tr, r_va):
    for col in frame:
        arr = frame[col].to_numpy(copy=True)
        with np.errstate(invalid='ignore'):
            arr[np.isnan(arr)] = np.nan
        frame[col] = arr
r_scores, r_mask, r_reconstruction = compare_imputers(r_tr, r_va)
r_median = SimpleImputer(strategy='median').fit(r_tr[impute_cols])
r_imputed_py = {s:pd.DataFrame(r_median.transform(d[impute_cols]),columns=impute_cols)
                for s,d in [('train',r_tr),('validation',r_va)]}
with localconverter(default_converter + pandas2ri.converter):
    ro.globalenv['imputer_scores_r'] = ro.conversion.py2rpy(r_scores)
    ro.globalenv['imputed_train_r'] = ro.conversion.py2rpy(r_imputed_py['train'])
    ro.globalenv['imputed_validation_r'] = ro.conversion.py2rpy(r_imputed_py['validation'])
    ro.globalenv['artificial_mask_r'] = ro.conversion.py2rpy(pd.DataFrame(r_mask[:,:2],columns=num_cols))
# Independently check R preprocessing against Python, including missing time values.
np.testing.assert_allclose(r_tr[impute_cols], clean['train'][impute_cols], equal_nan=True, atol=1e-12)
np.testing.assert_allclose(r_va[impute_cols], clean['validation'][impute_cols], equal_nan=True, atol=1e-12)
assert np.array_equal(r_mask, artificial_mask)
print('R and Python: matching numerical inputs and experimental masks.')

R and Python: matching numerical inputs and experimental masks.


In [14]:
%%R
print(imputer_scores_r,row.names=FALSE)
cat('Artificially hidden values:\n'); print(colSums(artificial_mask_r))
imputed_r <- list(train=imputed_train_r,validation=imputed_validation_r)

           method

             feature

  unit

 n_hidden

       MAE

           median

... (30 outputs left out)

   customer_age_num 

order_value_eur_num 

                 13 

                 14 

## R: 6. IQR review and decisions
Thresholds are fitted on training data only. Domain violations receive a separate reason; statistically unusual but domain-valid observations are retained.

In [15]:
%%R
review_r <- data.frame(); thresholds_r <- list()
for(s in names(clean_r)) {
  d <- clean_r[[s]]; d$message_length <- nchar(d$message_clean,type='chars')
  d$message_length[d$message_missing==1] <- NA_real_; clean_r[[s]] <- d
}
for(c in c('order_value_eur_num','message_length')) {
  q <- quantile(clean_r$train[[c]],c(.25,.75),na.rm=TRUE)
  bounds <- c(lower=q[1]-1.5*diff(q),upper=q[2]+1.5*diff(q)); names(bounds) <- c('lower','upper')
  thresholds_r[[c]] <- bounds
  for(s in names(clean_r)) {
    d <- clean_r[[s]]; flag <- !is.na(d[[c]]) & (d[[c]]<bounds[1] | d[[c]]>bounds[2])
    clean_r[[s]][[paste0(c,'_unusual')]] <- as.integer(flag)
    ix <- which(flag)
    if(length(ix)) review_r <- rbind(review_r,data.frame(split=s,ticket_id=d$ticket_id[ix],feature=c,
      value=d[[c]][ix],lower=bounds[1],upper=bounds[2],reason='IQR_review_potentially_valid',decision='keep'))
  }
}
for(s in names(clean_r)) for(c in c('customer_age','order_value_eur')) {
  d <- clean_r[[s]]; ix <- which(d[[paste0(c,'_domain_invalid')]]==1)
  if(length(ix)) review_r <- rbind(review_r,data.frame(split=s,ticket_id=d$ticket_id[ix],feature=c,
    value=d[[c]][ix],lower=if(c=='customer_age') 18 else 0,upper=if(c=='customer_age') 100 else Inf,
    reason='known_domain_violation',decision='working_value_to_missing'))
}
print(thresholds_r); print(review_r,row.names=FALSE)

$order_value_eur_num


   lower 

   upper 

-78.4475 

235.4525 

$message_length


... (361 outputs left out)

      Inf

       known_domain_violation

 working_value_to_missing

In addition: Warning messages:
1: In data.frame(split = s, ticket_id = d$ticket_id[ix], feature = c,  :
  row names were found from a short variable and have been discarded
2: In data.frame(split = s, ticket_id = d$ticket_id[ix], feature = c,  :
  row names were found from a short variable and have been discarded
3: In data.frame(split = s, ticket_id = d$ticket_id[ix], feature = c,  :
  row names were found from a short variable and have been discarded


## R: 7. Numerical scaling
The same transformation definitions are implemented directly in R. Standard deviation uses divisor n to match StandardScaler. Transformed validation values are not clipped to [0, 1].

In [16]:
%%R
fit_scaler <- function(x,method) {
  center <- switch(method,standard=colMeans(x),minmax=apply(x,2,min),robust=apply(x,2,median))
  spread <- switch(method,standard=sqrt(colMeans(sweep(x,2,colMeans(x))^2)),
    minmax=apply(x,2,function(z) max(z)-min(z)),robust=apply(x,2,IQR))
  spread[spread==0] <- 1
  list(center=center,spread=spread)
}
transform_scaler <- function(x,fit) sweep(sweep(as.matrix(x),2,fit$center),2,fit$spread,'/')
summary_numeric <- function(x) t(apply(as.matrix(x),2,function(z)
  c(n=length(z),mean=mean(z),sd_population=sqrt(mean((z-mean(z))^2)),min=min(z),
    q1=quantile(z,.25),median=median(z),q3=quantile(z,.75),max=max(z))))
tr <- as.matrix(imputed_r$train[,num_cols_r]); va <- as.matrix(imputed_r$validation[,num_cols_r])
cat('Before transformation - train\n'); print(summary_numeric(tr))
cat('Before transformation - validation\n'); print(summary_numeric(va))
scaled_r <- list(); scale_influence_r <- data.frame()
max_ix <- which.max(tr[,2]); keep <- setdiff(seq_len(nrow(tr)),max_ix)
for(method in c('standard','minmax','robust')) {
  fit <- fit_scaler(tr,method); alt <- fit_scaler(tr[keep,,drop=FALSE],method)
  scaled_r[[method]] <- transform_scaler(va,fit)
  for(s in c('train','validation')) {
    cat(method,s,'\n'); print(summary_numeric(transform_scaler(if(s=='train') tr else va,fit)))
  }
  cat('Transformed validation values:',method,'\n')
  print(data.frame(ticket_id=clean_r$validation$ticket_id,scaled_r[[method]]),row.names=FALSE)
  shift <- colMeans(abs(transform_scaler(tr[keep,,drop=FALSE],fit)-transform_scaler(tr[keep,,drop=FALSE],alt)))
  scale_influence_r <- rbind(scale_influence_r,data.frame(method=method,feature=names(shift),mean_abs_shift=shift))
}
outside_r <- sweep(va,2,apply(tr,2,min),'<') | sweep(va,2,apply(tr,2,max),'>')
cat('Validation values outside the training range:\n')
print(data.frame(ticket_id=clean_r$validation$ticket_id,va,outside_r)[rowSums(outside_r)>0,],row.names=FALSE)
print(scale_influence_r,row.names=FALSE)

Before transformation - train


   n

      mean

 sd_population

   min

 q1.25%

... (979 outputs left out)

    0.000000000

   robust

 order_value_eur_num

    0.034369752

## R: 8. Nominal encoding and ordered priority
Sparse one-hot matrices use a vocabulary learned on training data only. Both unknown-category policies use the same frequency threshold of 30. Examples show unseen validation cities and the chat channel.

In [17]:
%%R
nominal_cols_r <- c('channel_clean_explicit','city_clean_explicit')
fit_ohe <- function(d,infrequent=FALSE,min_frequency=30) {
  fields <- lapply(nominal_cols_r,function(c) {
    counts <- table(d[[c]]); rare <- if(infrequent) names(counts)[counts<min_frequency] else character()
    levels <- c(setdiff(sort(names(counts)),rare),if(length(rare)) '__INFREQUENT__')
    list(column=c,known=names(counts),rare=rare,levels=levels)
  })
  list(fields=fields,infrequent=infrequent,
    names=unlist(lapply(fields,function(f) paste0(f$column,'_',f$levels)),use.names=FALSE))
}
transform_ohe <- function(d,fit) {
  ii <- jj <- integer(); offset <- 0L
  for(f in fit$fields) {
    x <- d[[f$column]]
    if(length(f$rare)) x[x %in% f$rare | (!x %in% f$known & fit$infrequent)] <- '__INFREQUENT__'
    j <- match(x,f$levels); ok <- which(!is.na(j))
    ii <- c(ii,ok); jj <- c(jj,j[ok]+offset); offset <- offset+length(f$levels)
  }
  sparseMatrix(i=ii,j=jj,x=rep(1,length(ii)),dims=c(nrow(d),length(fit$names)),dimnames=list(NULL,fit$names))
}
encoders_r <- list(); encoded_r <- list()
for(policy in c('ignore','infrequent_if_exist')) {
  fit <- fit_ohe(clean_r$train,policy=='infrequent_if_exist')
  encoders_r[[policy]] <- fit
  encoded_r[[policy]] <- lapply(clean_r,transform_ohe,fit=fit)
  stopifnot(ncol(encoded_r[[policy]]$train)==ncol(encoded_r[[policy]]$validation))
  cat(policy,'feature names:\n'); print(fit$names)
  for(f in fit$fields) cat(f$column,'bucket exists:',length(f$rare)>0,'rare:',f$rare,'\n')
  unknown <- Reduce('|',lapply(nominal_cols_r,function(c) !clean_r$validation[[c]] %in% clean_r$train[[c]]))
  ix <- head(which(unknown),8)
  print(cbind(clean_r$validation[ix,c('ticket_id',nominal_cols_r)],
              as.matrix(encoded_r[[policy]]$validation[ix,,drop=FALSE])),row.names=FALSE)
}
for(s in names(clean_r)) {
  clean_r[[s]]$priority_code <- match(clean_r[[s]]$priority_clean,c('low','medium','high'))-1L
  clean_r[[s]]$priority_code[is.na(clean_r[[s]]$priority_code)] <- -1L
}
print(head(clean_r$validation[,c('ticket_id','priority','priority_clean','priority_code','priority_missing')],10))

ignore

feature names:


[1]

 "channel_clean_explicit___MISSING__"

 "channel_clean_explicit_email"      

 "channel_clean_explicit_phone"      

... (287 outputs left out)

   medium

         medium

             1

                0

## R: 9. TF-IDF and sparse final inputs
The R branch computes TF-IDF directly using `Matrix`: raw term frequency times smoothed IDF `log((1+n)/(1+df))+1`, followed by row-wise L2 normalization. Tokens contain at least two alphanumeric or underscore characters; unigrams, bigrams, and negations follow the Python approach. Illustrative phrases are transformed only.

In [18]:
%%R
tokens_r <- function(text,bigrams=FALSE) {
  text <- tolower(text)
  words <- regmatches(text,gregexpr('[[:alnum:]_]{2,}',text,perl=TRUE))[[1]]
  if(bigrams && length(words)>1) c(words,paste(head(words,-1),tail(words,-1))) else words
}
fit_tfidf <- function(text,bigrams=FALSE) {
  docs <- lapply(text,tokens_r,bigrams=bigrams)
  vocab <- sort(unique(unlist(docs)))
  df <- tabulate(match(unlist(lapply(docs,unique)),vocab),nbins=length(vocab))
  list(vocab=vocab,idf=log((1+length(text))/(1+df))+1,bigrams=bigrams)
}
transform_tfidf <- function(text,fit) {
  docs <- lapply(text,tokens_r,bigrams=fit$bigrams)
  ii <- jj <- integer(); xx <- numeric()
  for(i in seq_along(docs)) {
    index <- match(docs[[i]],fit$vocab); index <- index[!is.na(index)]
    if(length(index)) {
      counts <- table(index); j <- as.integer(names(counts))
      values <- as.numeric(counts)*fit$idf[j]; values <- values/sqrt(sum(values^2))
      ii <- c(ii,rep(i,length(j))); jj <- c(jj,j); xx <- c(xx,values)
    }
  }
  sparseMatrix(i=ii,j=jj,x=xx,dims=c(length(text),length(fit$vocab)),dimnames=list(NULL,fit$vocab))
}
phrases_r <- c('payment problem','no payment problem','order has arrived','order has not arrived')
text_models_r <- list(); text_matrices_r <- list(); text_stats_r <- data.frame()
for(name in c('unigram','unigram_bigram')) {
  fit <- fit_tfidf(clean_r$train$message_clean,name=='unigram_bigram')
  text_models_r[[name]] <- fit
  mats <- lapply(clean_r,function(d) transform_tfidf(d$message_clean,fit))
  text_matrices_r[[name]] <- mats
  for(s in names(mats)) {
    m <- mats[[s]]
    text_stats_r <- rbind(text_stats_r,data.frame(representation=name,split=s,rows=nrow(m),
       columns=ncol(m),nnz=nnzero(m),zero_vectors=sum(rowSums(m!=0)==0)))
  }
  probe <- transform_tfidf(phrases_r,fit); selected <- which(colSums(probe!=0)>0)
  cat(name,'illustrative phrases:\n'); print(round(as.matrix(probe[,selected,drop=FALSE]),4))
  cat('Negation:\n'); print(grep('(^| )(no|not)($| )',fit$vocab,value=TRUE))
}
print(text_stats_r,row.names=FALSE)
flag_cols_r <- c('message_missing','channel_missing','priority_missing','created_invalid','created_missing',
  as.vector(t(outer(c('customer_age','order_value_eur'),c('_missing','_parse_failed','_domain_invalid'),paste0))))
final_scaler_r <- fit_scaler(as.matrix(imputed_r$train[,num_cols_r]),'robust')
X_r <- list()
for(s in names(clean_r)) {
  d <- clean_r[[s]]
  dense <- cbind(transform_scaler(as.matrix(imputed_r[[s]][,num_cols_r]),final_scaler_r),
     as.matrix(imputed_r[[s]][,time_cols_r]),as.matrix(d[,c(flag_cols_r,'priority_code')]))
  X_r[[s]] <- cbind(Matrix(dense,sparse=TRUE),encoded_r$ignore[[s]],text_matrices_r$unigram_bigram[[s]])
  stopifnot(inherits(X_r[[s]],'sparseMatrix'),all(is.finite(X_r[[s]]@x)))
  cat(s,'final shape:',dim(X_r[[s]]),'\n')
}
stopifnot(ncol(X_r$train)==ncol(X_r$validation))
audit_r <- unique(audit_r)
print(aggregate(ticket_id~split+column+issue_type,audit_r,length),row.names=FALSE)

unigram

illustrative phrases:


 arrived

    has

     no

    not

... (253 outputs left out)

 validation

         message

             whitespace_only

         3

## Consistency checks, exports, and conclusions
Check independent R cleaning, UTC conversions, IQR thresholds, scaling, encoders, and numerical TF-IDF values. Exports contain the complete audit, review table, category frequencies, scores, and sparse matrices for further work. Features and identifiers are exported separately.

`chat` and `liberec` are new validation categories. Statistically unusual amounts and long reports are retained. MinMax-transformed validation values may fall outside [0, 1], indicating a range shift. Unigrams preserve `no`/`not`; bigrams additionally capture local context such as `not arrived`. Most-frequent imputation increases the most common channel's share; explicit missingness does not. Imputation scores describe a reconstruction experiment in years and euros, not classifier accuracy.

In [19]:
# Helper conversions are used for verification only, not for fitting R models.
def r_df(expr):
    with localconverter(default_converter + pandas2ri.converter):
        return ro.conversion.rpy2py(ro.r(expr))
for s in clean:
    d = clean[s]
    checked = r_df(f"clean_r${s}[,c('channel_clean_explicit','city_clean_explicit','priority_clean_explicit','priority_code')]")
    expected = d[['channel_clean_explicit','city_clean_explicit','priority_clean_explicit','priority_code']]
    for c in expected:
        assert checked[c].astype(str).tolist() == expected[c].astype(str).tolist(), (s,c)
    for c in ['created_at_utc','first_reply_at_utc']:
        seconds_r = np.array(ro.r(f'as.numeric(clean_r${s}${c})'),dtype=float)
        seconds_py = d[c].map(lambda z: z.timestamp() if pd.notna(z) else np.nan).to_numpy()
        np.testing.assert_allclose(seconds_r,seconds_py,equal_nan=True, atol=1e-12)
for c,bounds in thresholds.items():
    np.testing.assert_allclose(np.array(ro.r(f'thresholds_r[["{c}"]]')),bounds)
for name in scaled_validation:
    np.testing.assert_allclose(np.array(ro.r(f'scaled_r[["{name}"]]')),scaled_validation[name],atol=1e-12)
for policy in encoded:
    rn = list(ro.r(f'encoders_r[["{policy}"]]$names'))
    pn = encoders[policy].get_feature_names_out().tolist()
    # R uses its own display name for the infrequent bucket.
    rn = [v.replace('__INFREQUENT__','infrequent_sklearn') for v in rn]
    assert rn == pn
    for s in clean:
        np.testing.assert_allclose(np.array(ro.r(f'as.matrix(encoded_r[["{policy}"]]${s})')),
                                   encoded[policy][s].toarray())
for name in text_models:
    rn = list(ro.r(f'text_models_r[["{name}"]]$vocab'))
    pn = text_models[name].get_feature_names_out().tolist()
    assert set(rn) == set(pn)
    reorder = [rn.index(x) for x in pn]
    for s in clean:
        rm = np.array(ro.r(f'as.matrix(text_matrices_r[["{name}"]]${s})'))[:,reorder]
        np.testing.assert_allclose(rm,text_matrices[name][s].toarray(),atol=1e-12)
audit_table.to_csv(OUT/'audit_python.csv',index=False,encoding='utf-8-sig')
review.to_csv(OUT/'review_python.csv',index=False,encoding='utf-8-sig')
freq_table.to_csv(OUT/'category_frequencies_python.csv',index=False,encoding='utf-8-sig')
imputer_scores.to_csv(OUT/'imputation_scores.csv',index=False)
pd.DataFrame(text_stats).to_csv(OUT/'tfidf_summary.csv',index=False)
pd.DataFrame({'feature':feature_names_final}).to_csv(OUT/'feature_names_python.csv',index=False)
scale_summary.to_csv(OUT/'scaling_summary_python.csv')
scale_influence.to_csv(OUT/'scaling_influence_python.csv',index=False)
pd.DataFrame(artificial_mask[:,:2],columns=num_cols).assign(ticket_id=clean['validation'].ticket_id).to_csv(OUT/'artificial_validation_mask.csv',index=False)
for s,d in clean.items():
    sparse.save_npz(OUT/f'X_{s}_python.npz',X[s])
    d.to_csv(OUT/f'cleaned_{s}_python.csv',index=False,encoding='utf-8-sig')
    d[['ticket_id','customer_id','needs_escalation']].to_csv(OUT/f'identifiers_target_{s}.csv',index=False)
ro.globalenv['output_dir'] = str(OUT).replace('\\','/')
print('All Python/R consistency checks passed. Outputs:', OUT)
display(imputer_scores.pivot(index='method',columns='unit',values='MAE').rename(columns={'years':'Age MAE [years]','EUR':'Amount MAE [EUR]'}))

All Python/R consistency checks passed. Outputs:

 D:\Programko\UJEP\PYR\22_10_2026\vystup


unit,Amount MAE [EUR],Age MAE [years]
method,,
IterativeImputer,276.163665,16.426010
median,90.075714,16.307692
scaled_KNN,520.482286,14.107692


In [20]:
%%R
write.csv(audit_r,file.path(output_dir,'audit_R.csv'),row.names=FALSE,fileEncoding='UTF-8',na='')
write.csv(review_r,file.path(output_dir,'review_R.csv'),row.names=FALSE,fileEncoding='UTF-8',na='')
write.csv(frequencies_r,file.path(output_dir,'category_frequencies_R.csv'),row.names=FALSE,fileEncoding='UTF-8')
for(s in names(clean_r)) {
  write.csv(clean_r[[s]],file.path(output_dir,paste0('cleaned_',s,'_R.csv')),row.names=FALSE,fileEncoding='UTF-8',na='')
  saveRDS(X_r[[s]],file.path(output_dir,paste0('X_',s,'_R.rds')))
}
saveRDS(list(scaler=final_scaler_r,encoder=encoders_r$ignore,tfidf=text_models_r$unigram_bigram),
        file.path(output_dir,'preprocessing_R.rds'))
cat('R export completed.\n')

R export completed.


### Observed results

With the fixed mask (13 ages and 14 amounts):

| Method | Age MAE [years] | Amount MAE [EUR] |
|---|---:|---:|
| Median | 16.31 | 90.08 |
| Scaled KNN | 14.11 | 520.48 |
| IterativeImputer | 16.43 | 276.16 |

KNN has the lowest age error in this experiment; median imputation has the lowest amount error. This small masked sample does not establish a general ranking. Median imputation was chosen as a simple baseline, without tuning on validation labels.

Unigram TF-IDF shapes are **240 × 50** / **80 × 50**; unigram/bigram shapes are **240 × 113** / **80 × 113** (train / validation). Both representations have 31 zero training vectors and 10 zero validation vectors.

At threshold 30, the channel field has an infrequent bucket (phone and missing), but **the city field does not**. Under `infrequent_if_exist`, unknown `chat` maps to the infrequent bucket; `liberec` produces a zero block, as with `ignore`. Missing priority is always -1 with a separate indicator.

All R/Python consistency checks passed, including sparse TF-IDF values (tolerance 1e-12).